# Indian Traffic Sign Recognition (TSR) - Cloud GPU Training Pipeline
### Direct Cloud Dataset Download & YOLOv8n Training on Google Colab (Tesla T4 / L4)

This notebook provides a complete zero-local-storage workflow:
1. **Check GPU acceleration** (Tesla T4 / L4)
2. **Authenticate & Download real Indian Traffic Sign dataset** directly into Colab
3. **Configure `dataset.yaml`**
4. **Train YOLOv8n detector & classifier** (100 epochs, 640x640, AdamW)
5. **Evaluate metrics** (mAP@50, mAP@50-95, Precision, Recall, Confusion Matrix)
6. **Export `best.pt`** directly to Google Drive or download to your local `models/` directory

In [ ]:
# Step 1: Verify Colab GPU acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics YOLO and Kaggle CLI
!pip install --upgrade ultralytics opencv-python matplotlib kaggle

## Step 3: Download Real Indian Traffic Sign Dataset

Kaggle requires an API token to permit downloading datasets.

### How to get your free Kaggle API token in 15 seconds:
1. Go to **[kaggle.com](https://www.kaggle.com)** (log in or create a free account).
2. Click your **profile icon** in the top-right corner $\rightarrow$ Select **Settings**.
3. Scroll down to the **API** section $\rightarrow$ Click **Create New Token**.
4. A small file named `kaggle.json` will automatically download to your computer.
5. Run the cell below and select that `kaggle.json` file when prompted!

In [ ]:
# Step 3A: Upload kaggle.json to authenticate
import os, shutil
from google.colab import files

kaggle_dir = os.path.expanduser("~/.kaggle")
kaggle_json = os.path.join(kaggle_dir, "kaggle.json")

if not os.path.exists(kaggle_json):
    print("Please upload your 'kaggle.json' file:")
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        os.makedirs(kaggle_dir, exist_ok=True)
        shutil.move("kaggle.json", kaggle_json)
        os.chmod(kaggle_json, 0o600)
        print("Successfully configured kaggle.json!")
    else:
        print("Warning: No kaggle.json uploaded.")
else:
    print("kaggle.json already configured!")

In [ ]:
# Step 3B: Download and extract real Indian road traffic signs directly into Colab SSD
!kaggle datasets download -d kaustubhrastogi/traffic-signs-dataset-in-indian-roads -p /content/dataset --unzip

import os
if os.path.exists("/content/dataset"):
    print("\n--- Dataset Contents in /content/dataset ---")
    print(os.listdir("/content/dataset"))
else:
    print("Failed to download dataset. Check kaggle.json authentication.")

In [ ]:
# Step 4: Verify or Generate dataset.yaml configuration
import os, glob

# Search if dataset already includes a data.yaml or dataset.yaml
yaml_candidates = glob.glob("/content/dataset/**/*.yaml", recursive=True)

if yaml_candidates:
    dataset_yaml_path = yaml_candidates[0]
    print(f"Found dataset configuration file at: {dataset_yaml_path}")
    with open(dataset_yaml_path, 'r') as f:
        print("\n--- Existing YAML Configuration ---\n" + f.read())
else:
    dataset_yaml_path = "/content/dataset.yaml"
    yaml_content = """
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 15
names:
  0: Stop
  1: No Entry
  2: Speed Limit 20
  3: Speed Limit 40
  4: Speed Limit 60
  5: Speed Limit 80
  6: Give Way
  7: School Ahead
  8: Pedestrian Crossing
  9: Left Turn
  10: Right Turn
  11: U-Turn
  12: Horn Prohibited
  13: Parking
  14: Dangerous Curve
"""
    with open(dataset_yaml_path, "w") as f:
        f.write(yaml_content.strip())
    print(f"Created default configuration at: {dataset_yaml_path}")

In [ ]:
# Step 5: Mount Google Drive (to save models and persistent weights)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 6: Train YOLOv8n on Cloud GPU (Tesla T4)
from ultralytics import YOLO

# Load pretrained model weights
model = YOLO("yolov8n.pt")

# Train for 100 epochs (optimized for speed & accuracy)
results = model.train(
    data=dataset_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    device=0,
    project="tsr_colab_run",
    name="yolov8n_indian_tsr",
    save=True,
    plots=True
)

In [ ]:
# Step 7: Evaluate Model Metrics on Validation Set
metrics = model.val()
print(f"Validation Results on Real Indian Road Test Data:")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Step 8: Visualize Results and Confusion Matrix
from IPython.display import Image, display
import glob

results_png = glob.glob("/content/tsr_colab_run/yolov8n_indian_tsr/results.png")
if results_png:
    print("--- Training Loss & Metric Curves ---")
    display(Image(results_png[0]))

cm_png = glob.glob("/content/tsr_colab_run/yolov8n_indian_tsr/confusion_matrix.png")
if cm_png:
    print("--- Confusion Matrix ---")
    display(Image(cm_png[0]))

In [ ]:
# Step 9: Export best.pt to Google Drive & download to your laptop
import shutil
from google.colab import files

best_weights = "/content/tsr_colab_run/yolov8n_indian_tsr/weights/best.pt"
if os.path.exists(best_weights):
    # Save backup to Google Drive
    os.makedirs("/content/drive/MyDrive/TSR_Models", exist_ok=True)
    shutil.copy(best_weights, "/content/drive/MyDrive/TSR_Models/best.pt")
    print("Successfully saved best weights to Google Drive: /content/drive/MyDrive/TSR_Models/best.pt")
    
    # Download directly to your laptop's 'models/' folder
    files.download(best_weights)
else:
    print("best.pt was not found. Please review the training logs above.")